# Full Analysis - Context-Aware Prompting (N=261)

**Model:** gpt-5.4-mini-2026-03-17  
**Method:** Context-Aware Few-shot, Temperature=0  
**Dataset:** 3 domains  
- E-commerce / Sylius: 100 samples  
- Finance/Banking / Apache Fineract: 100 samples  
- Social Network / Diaspora: 61 samples

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
import os
from scipy.stats import wilcoxon
from scipy.stats import binomtest

sns.set_theme(style='whitegrid')
figures_dir = '../figures'
os.makedirs(figures_dir, exist_ok=True)

## 1. Load Data

In [ ]:
df = pd.read_csv('summary.csv')
print(f'Total samples: {len(df)}')
display(df.head())

## 2. Descriptive Statistics

In [ ]:
display(df[['raw_cosine', 'skeleton_cosine']].describe())
print('\nExecutable counts:')
display(df['executable'].value_counts())

## 3. Figure 1: Distribution of Skeleton Cosine Similarity

Histogram + Boxplot combo. RBL-4 compliant: title, axis labels, N annotation, 300 DPI.

In [ ]:
fig, (ax_box, ax_hist) = plt.subplots(2, 1, figsize=(10, 6),
                                       sharex=True,
                                       gridspec_kw={'height_ratios': [1, 4], 'hspace': 0.05})

sns.boxplot(x=df['skeleton_cosine'], color='mediumpurple', ax=ax_box)
ax_box.axvline(0.85, color='red', linestyle='--', linewidth=2)
ax_box.set_yticks([])
ax_box.set_title('Distribution of Skeleton Cosine Similarity (N=261)', fontsize=14)

ax_hist.hist(df['skeleton_cosine'], bins=20, color='mediumpurple', edgecolor='black', alpha=0.8)
ax_hist.axvline(0.85, color='red', linestyle='--', linewidth=2, label='Threshold = 0.85')
median_val = df['skeleton_cosine'].median()
ax_hist.axvline(median_val, color='orange', linestyle='-', linewidth=2, label=f'Median = {median_val:.4f}')
ax_hist.set_xlabel('Skeleton Cosine Similarity (all-MiniLM-L6-v2)', fontsize=12)
ax_hist.set_ylabel('Count', fontsize=12)
ax_hist.text(0.02, 0.92, f'N = {len(df)}', transform=ax_hist.transAxes, fontsize=12,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax_hist.legend(loc='upper left', fontsize=11)
ax_hist.set_xlim(0.3, 1.02)

plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'fig1_skeleton_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/fig1_skeleton_distribution.png')

## 4. Figure 2: Skeleton Cosine by Domain

Boxplot comparing skeleton cosine across 3 domains.

In [ ]:
# Merge with user stories to get domain column
df_us = pd.read_csv('../data/full_user_stories_261.csv')
df_merged = df.merge(df_us[['id', 'domain']], on='id', how='left')

fig, ax = plt.subplots(figsize=(10, 6))
domain_order = ['E-commerce', 'Finance/Banking', 'Social Network']
palette = {'E-commerce': '#5B8FF9', 'Finance/Banking': '#5AD8A6', 'Social Network': '#F6BD16'}
sns.boxplot(data=df_merged, x='domain', y='skeleton_cosine', order=domain_order, palette=palette, ax=ax)
ax.axhline(0.85, color='red', linestyle='--', linewidth=2, label='Threshold = 0.85')
ax.set_title('Skeleton Cosine Similarity by Domain (N=261)', fontsize=14)
ax.set_xlabel('Domain', fontsize=12)
ax.set_ylabel('Skeleton Cosine Similarity', fontsize=12)
ax.legend(loc='lower right', fontsize=11)

# Annotate median per domain
for i, domain in enumerate(domain_order):
    subset = df_merged[df_merged['domain'] == domain]['skeleton_cosine']
    med = subset.median()
    n = len(subset)
    ax.text(i, med + 0.01, f'Mdn={med:.3f}\nn={n}', ha='center', fontsize=10, fontweight='bold')

ax.text(0.02, 0.02, f'Total N = {len(df_merged)}', transform=ax.transAxes, fontsize=12,
        verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'fig2_domain_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/fig2_domain_comparison.png')

## 5. Figure 3: Executable Syntax Rate

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
exec_counts = df['executable'].value_counts()
bars = ax.bar(exec_counts.index, exec_counts.values, color=['#66b3ff', '#ff9999'], edgecolor='black', width=0.5)
ax.axhline(y=len(df)*0.80, color='red', linestyle='--', linewidth=2, label=f'Threshold 80% ({int(len(df)*0.80)} samples)')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1, f'{int(height)}\n({height/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Executable Syntax Rate (N=261)', fontsize=14)
ax.set_xlabel('Executable Status (behave --dry-run)')
ax.set_ylabel('Number of User Stories')
ax.text(0.02, 0.92, f'N = {len(df)}', transform=ax.transAxes, fontsize=12,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.legend(loc='upper right')
ax.set_ylim(0, max(exec_counts.values) + 30)
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'fig3_executable_rate.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/fig3_executable_rate.png')

## 6. Hypothesis Testing - RQ1: Semantic Similarity

- $H_0^{(1)}: \text{median}_{sim} \leq 0.85$
- $H_1^{(1)}: \text{median}_{sim} > 0.85$
- **Test:** One-sample Wilcoxon signed-rank test (one-tailed, $\alpha = 0.05$)

In [ ]:
alpha = 0.05
stat, p_val = wilcoxon(df['skeleton_cosine'] - 0.85, alternative='greater')
print(f'Wilcoxon Statistic: {stat}')
print(f'P-value: {p_val:.6f}')

if p_val < alpha:
    rq1_conclusion = 'Reject H0'
    print(f'\n=> CONCLUSION: Reject H0 (p={p_val:.6f} < {alpha}). There is sufficient statistical evidence that skeleton_cosine > 0.85.')
else:
    rq1_conclusion = 'Fail to reject H0'
    print(f'\n=> CONCLUSION: Fail to reject H0 (p={p_val:.6f} >= {alpha}). NOT enough evidence that skeleton_cosine > 0.85.')

## 7. Hypothesis Testing - RQ2: Executable Rate

- $H_0^{(2)}: p_{exec} \leq 0.80$
- $H_1^{(2)}: p_{exec} > 0.80$
- **Test:** Exact Binomial Test (one-tailed, $\alpha = 0.05$)

In [ ]:
k = len(df[df['executable'] == 'PASS'])
n = len(df)
res = binomtest(k, n, p=0.80, alternative='greater')
print(f'PASS count: {k}/{n} ({k/n:.2%})')
print(f'P-value: {res.pvalue:.6f}')

if res.pvalue < alpha:
    rq2_conclusion = 'Reject H0'
    print(f'\n=> CONCLUSION: Reject H0 (p={res.pvalue:.6f} < {alpha}). There is sufficient statistical evidence that executable rate > 80%.')
else:
    rq2_conclusion = 'Fail to reject H0'
    print(f'\n=> CONCLUSION: Fail to reject H0 (p={res.pvalue:.6f} >= {alpha}). NOT enough evidence that executable rate > 80%.')

## 8. API Cost Analysis

In [ ]:
logs = []
with open('api_logs_261.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        logs.append(json.loads(line))

df_logs = pd.DataFrame(logs)
print('='*50)
print('       API COST ANALYSIS (FULL)')
print('='*50)
print(f'  Total API calls:        {len(df_logs)}')
print(f'  Total prompt tokens:    {df_logs["prompt_tokens"].sum():,}')
print(f'  Total completion tokens: {df_logs["completion_tokens"].sum():,}')
print(f'  Total tokens:           {df_logs["total_tokens"].sum():,}')
print(f'  Total cost (USD):       ${df_logs["cost_usd"].sum():.4f}')
print(f'  Avg cost per call:      ${df_logs["cost_usd"].mean():.6f}')

## 9. Export summary.csv

In [ ]:
summary_data = [
    {
        'RQ': 'RQ1',
        'metric': 'skeleton_cosine',
        'threshold': 0.85,
        'observed_median': df['skeleton_cosine'].median(),
        'observed_mean': df['skeleton_cosine'].mean(),
        'N': len(df),
        'p_value': p_val,
        'conclusion': rq1_conclusion
    },
    {
        'RQ': 'RQ2',
        'metric': 'executable_rate',
        'threshold': 0.80,
        'observed_median': k/n,
        'observed_mean': k/n,
        'N': n,
        'p_value': res.pvalue,
        'conclusion': rq2_conclusion
    }
]

df_summary = pd.DataFrame(summary_data)
df_summary.to_csv('summary.csv', index=False)
display(df_summary)
print('\nSaved: results/summary.csv')